## This notebook was structured to provide a clear display over tables (purely for confort and reflection)

In [1]:
#import necessary libraries and set up data paths for the analysis. Do not edit this cell. Run first.
import duckdb
import pandas as pd
from pathlib import Path

data_path = Path("../data/raw")
visitors_path = (data_path / "visitors.parquet").as_posix()
sessions_path = (data_path / "sessions.parquet").as_posix()
events_path = (data_path / "questionnaire_events.parquet").as_posix()
answers_path = (data_path / "questionnaire_answers.parquet").as_posix()
questions_path = (data_path / "questionnaire_questions.parquet").as_posix()
outcomes_path = (data_path / "questionnaire_outcomes.parquet").as_posix()

In [2]:
##Cell to check the questionnaire
duckdb.sql(f"""
SELECT question_text, allowed_answers
FROM read_parquet('{questions_path}')
ORDER BY question_number
""").show()

┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                    question_text                                                     │                                        allowed_answers                                         │
│                                                       varchar                                                        │                                            varchar                                             │
├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────┤
│ Have you ever been told by a healthcare professional that you have Type 2 Diabetes?                                  │ ["yes",

In [3]:
# Cell to check the number of records in each of the most important tables in the dataset.

duckdb.sql(f"""
SELECT
    (SELECT COUNT(*) FROM read_parquet('{visitors_path}')) AS visitors,
    (SELECT COUNT(*) FROM read_parquet('{sessions_path}')) AS sessions,
    (SELECT COUNT(*) FROM read_parquet('{outcomes_path}')) AS outcomes
""").df()

,visitors,sessions,outcomes
0,20000,26000,11357


In [4]:
#Cell to display visitors table
display(duckdb.sql(f"""
SELECT *
FROM read_parquet('{visitors_path}')
LIMIT 5
""").df())

,visitor_id,birth_year,gender,region,first_seen_at
0,VIS_00000001,2008,male,Bretagne,2026-08-02 06:29:33.074952+02:00
1,VIS_00000002,1946,female,Hauts-de-France,2026-07-25 13:59:57.752213+02:00
2,VIS_00000003,<NA>,female,Pays de la Loire,2026-06-22 14:37:47.114315+02:00
3,VIS_00000004,1989,other,Provence-Alpes-Côte d'Azur,2026-07-12 04:29:05.408274+02:00
4,VIS_00000005,1989,female,Hauts-de-France,2026-07-10 07:33:03.390070+02:00


In [5]:
#Cell to display sessions table
display(duckdb.sql(f"""
SELECT *
FROM read_parquet('{sessions_path}')
LIMIT 5
""").df())

,session_id,visitor_id,session_started_at,acquisition_source,campaign_name,device_type,landing_page,is_returning_visitor
0,SES_00000001,VIS_00016128,2026-06-01 02:01:59.429044+02:00,social,diabetes_social_awareness,desktop,/health/diabetes-awareness,False
1,SES_00000002,VIS_00012147,2026-06-01 02:07:54.468178+02:00,google_organic,NaN,desktop,/health/diabetes-awareness,False
2,SES_00000003,VIS_00018597,2026-06-01 02:10:06.909833+02:00,google_ads,diabetes_awareness_general,desktop,/health/diabetes-awareness,False
3,SES_00000004,VIS_00006671,2026-06-01 02:14:47.814240+02:00,google_organic,NaN,mobile,/health/diabetes-awareness,False
4,SES_00000005,VIS_00018327,2026-06-01 02:14:55.885097+02:00,chatgpt,NaN,mobile,/health/diabetes-awareness,False


In [6]:
duckdb.sql(f"""
SELECT DISTINCT campaign_name
FROM read_parquet('{sessions_path}')
ORDER BY campaign_name
""").df()

,campaign_name
0,diabetes_45plus
1,diabetes_awareness_general
2,diabetes_retargeting
3,diabetes_social_awareness
4,NaN


In [7]:
duckdb.sql(f"""
SELECT DISTINCT acquisition_source
FROM read_parquet('{sessions_path}')
ORDER BY acquisition_source
""").df()

,acquisition_source
0,AI Assistant
1,ChatGPT
2,Google
3,chatgpt
4,direct
5,email
6,facebook
7,google
8,google_ads
9,google_organic


In [8]:
#Cell to display events table
display(duckdb.sql(f"""
SELECT *
FROM read_parquet('{events_path}')
LIMIT 5
""").df())

,event_id,session_id,visitor_id,event_timestamp,event_type,question_number
0,EVT_000000001,SES_00000001,VIS_00016128,2026-06-01 02:01:59.429044+02:00,landing_view,<NA>
1,EVT_000000002,SES_00000001,VIS_00016128,2026-06-01 02:02:07.868799+02:00,questionnaire_start,<NA>
2,EVT_000000003,SES_00000001,VIS_00016128,2026-06-01 02:02:11.208955+02:00,question_view,1
3,EVT_000000004,SES_00000001,VIS_00016128,2026-06-01 02:02:35.588484+02:00,question_answer,1
4,EVT_000000005,SES_00000001,VIS_00016128,2026-06-01 02:02:36.940089+02:00,question_view,2


In [9]:
#Cell to display answers table
display(duckdb.sql(f"""
SELECT *
FROM read_parquet('{answers_path}')
LIMIT 5
""").df())

,answer_id,session_id,visitor_id,question_id,question_number,answer_value,answered_at
0,ANS_000000001,SES_00000001,VIS_00016128,diabetes_q1,1,no,2026-06-01 02:02:35.588484+02:00
1,ANS_000000002,SES_00000002,VIS_00012147,diabetes_q1,1,no,2026-06-01 02:09:23.966499+02:00
2,ANS_000000003,SES_00000002,VIS_00012147,diabetes_q2,2,no,2026-06-01 02:09:48.773896+02:00
3,ANS_000000004,SES_00000003,VIS_00018597,diabetes_q1,1,no,2026-06-01 02:11:09.794015+02:00
4,ANS_000000005,SES_00000003,VIS_00018597,diabetes_q2,2,no,2026-06-01 02:11:33.897800+02:00


In [10]:
#Cell to display outcomes table
display(duckdb.sql(f"""
SELECT *
FROM read_parquet('{outcomes_path}')
LIMIT 5
""").df())

,session_id,visitor_id,outcome_category,completed_at
0,SES_00000003,VIS_00018597,no_current_indication,2026-06-01 02:12:52.331208+02:00
1,SES_00000004,VIS_00006671,no_current_indication,2026-06-01 02:16:49.679841+02:00
2,SES_00000008,VIS_00011878,possible_risk,2026-06-01 02:25:13.601379+02:00
3,SES_00000010,VIS_00013989,no_current_indication,2026-06-01 02:29:10.634712+02:00
4,SES_00000011,VIS_00002128,no_current_indication,2026-06-01 02:33:05.062401+02:00
